# Conjuntos experimentales

In [ ]:
# Importar bibliotecas necesarias
import json
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split

In [ ]:
# Leer el archivo de configuración
PROYECTO = Path("..")
ruta_config = PROYECTO / "config" / "dataset.json"
with ruta_config.open("r", encoding="utf-8") as archivo:
	config = json.load(archivo)

# Definir rutas de archivos y directorios
RUTA_MANIFEST = PROYECTO / config["manifest"]["path"]
DIRECTORIO_DATA = PROYECTO / "data"
DIRECTORIO_DATASET = PROYECTO / config["dataset"]["root"]
IMAGES = DIRECTORIO_DATASET / config["dataset"]["images"]

# Cargar el archivo manifest
manifest = pd.read_csv(RUTA_MANIFEST)
manifest.head()

In [ ]:
# Función para generar los splits de entrenamiento, validación y prueba
def generar_split(df, proporcion_train, semilla=42): # la semilla se usa para reproducibilidad

	# Primera partición: train y conjunto temporal
	train, temporal = train_test_split(
		df,
		train_size=proporcion_train,
		stratify=df["breed"], # Procura que la distribución de razas sea similar en ambos conjuntos
		random_state=semilla
	)

	# Segunda partición: temporal en validation y test
	validation, test = train_test_split(
		temporal,
		test_size=0.5,
		stratify=temporal["breed"], # Procura que la distribución de razas sea similar en ambos conjuntos
		random_state=semilla
	)

	# Copiar dataframes para después añadir la columna "split" sin modificar los originales
	train_set = train.copy()
	validation_set = validation.copy()
	test_set = test.copy()

	train_set["split"] = "train"
	validation_set["split"] = "validation"
	test_set["split"] = "test"

	# Reunir las particiones
	resultado = pd.concat(
		[train_set, validation_set, test_set],
		ignore_index=True
	)

	return resultado

In [ ]:
# Generar los splits con las proporciones deseadas
split_70 = generar_split(manifest, proporcion_train=0.70)
split_80 = generar_split(manifest, proporcion_train=0.80)

# Columnas que queremos conservar
columnas = ["image_id", "breed", "image_path", "mask_path", "split"]

# Guardar CSV
directorio_splits = DIRECTORIO_DATA  / "splits"
directorio_splits.mkdir(parents=True, exist_ok=True)

# Guardar los splits en archivos CSV
split_70[columnas].to_csv(directorio_splits / "split_70_15_15.csv", index=False)
split_80[columnas].to_csv( directorio_splits / "split_80_10_10.csv", index=False)

# Actualizar archivo de configuración con la ruta de los splits
with open(ruta_config, "r", encoding="utf-8") as f:
	config = json.load(f)

config["splits"] = {
	"path": "data/splits"
}

with open(ruta_config, "w", encoding="utf-8") as f:
	json.dump(config, f, indent=4, ensure_ascii=False)

In [ ]:
# Mostrar la distribución de razas en cada split
for nombre, datos in [
	("70/15/15", split_70),
	("80/10/10", split_80)
]:
	print(f"\nPartición {nombre}")

	print(pd.crosstab(datos["breed"], datos["split"])
	)

In [ ]:
# Verificar que los splits contienen todos los IDs de imagen del manifest y que no hay duplicados
for nombre, datos in [
	("70/15/15", split_70),
	("80/10/10", split_80)
]:
	# Continuar si las verificaciones son correctas, sino lanzar un error
	assert datos["image_id"].is_unique, (f"Error: existen imágenes duplicadas en {nombre}") 
	assert set(datos["image_id"]) == set(manifest["image_id"]), (f"Error: los identificadores no coinciden en {nombre}")

	print(f"{nombre}: partición verificada correctamente")

In [ ]:
# Prueba de carga de los splits desde los archivos CSV
directorio_splits = PROYECTO / config["splits"]["path"]

split_70 = pd.read_csv(directorio_splits / "split_70_15_15.csv")
print(f"Split 70/15/15 cargado con {len(split_70)} registros.")

split_80 = pd.read_csv(directorio_splits / "split_80_10_10.csv")
print(f"Split 80/10/10 cargado con {len(split_80)} registros.")